In [1]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

output_dir = '/content/drive/MyDrive/omg_diploma_2025/data'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)


Mounted at /content/drive


In [2]:
!pip install -q datasets pandas conllu tqdm nltk

from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import json
import pandas as pd
import conllu
import requests
import numpy as np
import ast
import re
import random
import zipfile
import io
import gc
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# datasets ideas (for start)
# https://huggingface.co/datasets/IlyaGusev/ru_news
# https://huggingface.co/datasets/IlyaGusev/gazeta (https://arxiv.org/pdf/2006.11063)
# https://huggingface.co/datasets/RussianNLP/rucola (https://github.com/IlyaGusev/RuCoLA/tree/main) # for evaluation
# https://huggingface.co/datasets/Helsinki-NLP/opus_books
# https://huggingface.co/datasets/universal-dependencies/universal_dependencies (https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master)


# print("Loading Lenta.ru...")
# dataset = load_dataset("ilearning/lenta_ru_news", split="train", streaming=True)


### google drive setup

# Text cleaning

In [4]:
MAX_WORDS_PER_CHUNK = 250
# TARGET_COUNT = 15000
# WIKI_LIMIT = 200000
TARGET_PER_SOURCE = 60000

LABEL_MAP = {
    "O": 0, # Space only
    ".": 1,
    ",": 2,
    "!": 3,
    "?": 4,
    ":": 5,
    ";": 6,
    "-": 7,

    '"': 8,
    ' "': 9,
    '" ': 10,

    ', "': 11,
    ': "': 12,
    '. "': 13,

    '",': 14,
    '".': 15,
    '"?': 16,
    '"!': 17,
    '...': 18,

    '- "': 19,
    '", -': 20, # direct speech with closing "
    '!" -': 21,
    '?" -': 22,
    '. -': 23,

    '""': 24,      # Double close (поступил в учреждение "школа "Эврика"".)

    "! -": 25, # direct speech without closing "
    "? -": 26,
    ", -": 27,
}

ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}


In [5]:
for k, v in LABEL_MAP.items(): print(k, v)


O 0
. 1
, 2
! 3
? 4
: 5
; 6
- 7
" 8
 " 9
"  10
, " 11
: " 12
. " 13
", 14
". 15
"? 16
"! 17
... 18
- " 19
", - 20
!" - 21
?" - 22
. - 23
"" 24
! - 25
? - 26
, - 27


In [6]:
file_path = os.path.join(output_dir, "label_map.json")
with open(file_path, 'w', encoding='utf-8') as f:
    json.dump(ID_TO_LABEL, f, ensure_ascii=False)


In [7]:
def robust_clean_text(text):
    text = re.sub(r'[\u2010-\u2015]', '-', text)
    text = re.sub(r'[«»“”„]', '"', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [8]:
def get_canonical_label(gap_text):
    if not gap_text.strip():
        return "O"

    # capture spacing before stripping to distinguishes ' "' from '" '
    has_leading_space = gap_text.startswith(' ')
    # assume standard punctuation implies trailing space for simple marks
    # but for quotes it's different
    skeleton = re.sub(r'\s+', '', gap_text)

    if skeleton == '.': return '.'
    if skeleton == ',': return ','
    if skeleton == '!': return '!'
    if skeleton == '?': return '?'
    if skeleton == ':': return ':'
    if skeleton == ';': return ';'
    if skeleton == '-': return '-'

    if skeleton == '"':
        # If leading space => Open Quote
        if has_leading_space:
            return ' "'
        # otherwise assume Close Quote
        return '" '

    if skeleton == ',"': return ', "'
    if skeleton == ':"': return ': "'
    if skeleton == '."': return '. "'

    if skeleton == '",': return '",'
    if skeleton == '".': return '".'
    if skeleton == '"?': return '"?'
    if skeleton == '"!': return '"!'

    if skeleton == '-"': return '- "'
    if skeleton == '",-': return '", -'
    if skeleton == '!"-': return '!"-'
    if skeleton == '?"-': return '?"-'

    if skeleton == '!-': return '! -'
    if skeleton == '?-': return '? -'
    if skeleton == '.-': return '. -'
    if skeleton == ',-': return ', -'

    if skeleton == '""': return '""' # Fallback
    if skeleton == '"",': return '"",'

    return "O"


In [9]:

def extract_labels_standardized(text):
    text = robust_clean_text(text)

    # number:
    # -?              : Optional negative
    # \d+             : digit start
    # (?:[.,]\d+)+    : followed by a group of (dot OR comma) + digits, repeated 1+ times
    # punct doesn't divide parts of number
    number_pattern = r"-?\d+(?:[.,]\d+)+"

    # word:
    # -?           -> optional minus (in numbers like -1.5)
    # [(\[]?       -> Optional ( or [
    # [\w]+        -> word characters
    # (?:-[\w]+)*  -> Optional hyphen
    # [)\]]?       -> Optional closing ) or ]
    word_pattern = r"-?[(\[]?[\w]+(?:-[\w]+)*[)\]]?"

    # standalone brackets
    brackets_pattern = r"[()]|[\[\]]"

    full_pattern = f"{number_pattern}|{word_pattern}|{brackets_pattern}"

    word_iter = list(re.finditer(full_pattern, text))

    result_pairs = []

    for i, match in enumerate(word_iter):
        word = match.group()
        start_gap = match.end()

        if i + 1 < len(word_iter):
            end_gap = word_iter[i+1].start()
            raw_gap = text[start_gap:end_gap]
        else:
            raw_gap = text[start_gap:]

        label_str = get_canonical_label(raw_gap)
        label_id = LABEL_MAP.get(label_str, 0)

        result_pairs.append({"word": word, "label": label_id})

    return result_pairs

In [10]:
def create_json_dataset(texts, output_filename):
    processed_data = []
    print(f"Processing {len(texts)} texts for {output_filename}...")

    for i, text in enumerate(tqdm(texts)):
        try:
            pairs = extract_labels_standardized(text)
        except:
            continue

        if len(pairs) < 5: continue # Skip tiny fragments

        tokens = [p['word'] for p in pairs]
        ner_tags = [p['label'] for p in pairs]

        processed_data.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })

    full_path = os.path.join(output_dir, output_filename)

    with open(full_path, 'w', encoding='utf-8') as f:
        json.dump(processed_data, f, ensure_ascii=False)

    del processed_data
    gc.collect()

    print(f"Saved {full_path}")

In [11]:
def fix_wiki_encoding(text):
    if not isinstance(text, str):
        # If bytes, decode it
        try:
            text = text.decode('utf-8')
        except:
            pass
    if text.startswith("b'") or text.startswith('b"'):
        try:
            # literal_eval turns "b'\xd0\xb0'" (str) -> b'\xd0\xb0' (bytes)
            # then .decode('utf-8') turns bytes -> "а" (str)
            text = ast.literal_eval(text).decode('utf-8')
        except (ValueError, SyntaxError):
            pass

    tags_to_remove = [
        "_START_ARTICLE_",
        "_START_SECTION_",
        # "_START_PARAGRAPH_",
        # "_NEWLINE_"
    ]

    for tag in tags_to_remove:
        text = text.replace(tag, " ")

    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [12]:
def chunk_external_text(text, max_words=MAX_WORDS_PER_CHUNK):
    sentences = sent_tokenize(text, language='russian')
    chunks = []
    current_chunk_sentences = []
    current_word_count = 0

    for sentence in sentences:
        sentence_words = sentence.split()
        sentence_len = len(sentence_words)

        if current_chunk_sentences and (current_word_count + sentence_len > max_words):
            chunks.append(" ".join(current_chunk_sentences))
            current_chunk_sentences = [sentence]
            current_word_count = sentence_len
        else:
            current_chunk_sentences.append(sentence)
            current_word_count += sentence_len

    if current_chunk_sentences:
        chunks.append(" ".join(current_chunk_sentences))

    return chunks

# Syntagrus preparation

In [13]:
# URLs for training
syntagrus_urls = [
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-a.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-b.conllu",
    "https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-train-c.conllu"
]


In [14]:
def process_syntagrus_files(urls, dataset_name):
    chunks = []
    print(f"--- PROCESSING {dataset_name} ---")
    for url in urls:
        fname = url.split("/")[-1]
        if not os.path.exists(fname):
            r = requests.get(url)
            with open(fname, "wb") as f: f.write(r.content)

        current_chunk = []
        current_word_count = 0
        current_doc_id = None
        current_genre = None

        with open(fname, "r", encoding="utf-8") as f:
            for sentence in tqdm(conllu.parse_incr(f)):
                sent_id = sentence.metadata['sent_id']
                text = sentence.metadata['text']
                genre = sentence.metadata.get('genre', 'unknown')
                doc_id = sent_id.rsplit("_", 1)[0] if "_" in sent_id else sent_id

                if doc_id != current_doc_id:
                    if current_chunk:
                        chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
                    current_chunk = []
                    current_word_count = 0
                    current_doc_id = doc_id
                    current_genre = genre
                elif current_genre == 'unknown' and genre != 'unknown':
                    current_genre = genre

                sent_len = len(text.split())
                if current_chunk and (current_word_count + sent_len > MAX_WORDS_PER_CHUNK):
                    chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
                    current_chunk = []
                    current_word_count = 0
                current_chunk.append(text)
                current_word_count += sent_len

        if current_chunk:
            chunks.append({"text": " ".join(current_chunk), "genre": current_genre})
    return pd.DataFrame(chunks)


In [15]:
df_syn = process_syntagrus_files(syntagrus_urls, "SynTagRus TRAIN")


--- PROCESSING SynTagRus TRAIN ---


24516it [00:28, 845.39it/s] 
24299it [00:12, 1974.77it/s]
20816it [00:08, 2577.90it/s]


In [16]:
# for url in syntagrus_urls:
#     fname = url.split("/")[-1]
#     if not os.path.exists(fname):
#         r = requests.get(url)
#         with open(fname, "wb") as f: f.write(r.content)

#     with open(fname, "r", encoding="utf-8") as f:
#         # SynTagRus is already sentence-split, so we just grab them
#         for sentence in conllu.parse_incr(f):
#             txt = sentence.metadata['text']
#             train_texts.append(txt)

# print(f"SynTagRus Size: {len(train_texts)} sentences")


# Gazeta load

In [17]:
# ext_journalism = []
# gazeta_url = "https://huggingface.co/datasets/IlyaGusev/gazeta/resolve/main/gazeta_train.jsonl"

# response = requests.get(gazeta_url, stream=True)
# total_size = int(response.headers.get('content-length', 0))
# line_count = 0
# # target_count = 15000

# for line in tqdm(response.iter_lines(), desc="Gazeta Lines"): # , total=target_count
#     # if line_count >= target_count:
#     #     break
#     if line:
#         try:
#             data = json.loads(line)
#             # 'text' is the article body
#             t = data['text'].replace('\n', ' ')
#             if len(t) > 100:
#                 ext_journalism.append(t)
#                 line_count += 1
#         except:
#             continue

# print(f"Loaded {len(ext_journalism)} Gazeta samples.")

In [18]:
ext_journalism =[]
ds_gazeta = load_dataset("IlyaGusev/gazeta", split="train", streaming=True)

count = 0
for item in tqdm(ds_gazeta, desc="Gazeta Stream"):
    t = item['text'].replace('\n', ' ')
    if len(t) > 100:
        chunks = chunk_external_text(t)
        ext_journalism.extend(chunks)
        count += 1
        if count >= 15000:
            break

print(f"Loaded {len(ext_journalism)} Gazeta articles.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Gazeta Stream: 14999it [00:39, 375.36it/s]

Loaded 43513 Gazeta articles.


# Helsinki Opus books

In [19]:
# from oxen import RemoteRepo
# repo = RemoteRepo("Helsinki-NLP/opus_books")
# repo.download("en-ru/opus_books_en-ru_train.parquet", revision="main")

# import pandas as pd
# df = pd.read_parquet("https://hub.oxen.ai/api/repos/Helsinki-NLP/opus_books/file/main/en-ru/opus_books_en-ru_train.parquet")
# print(df.head())

# "https://opus.nlpl.eu/Books/ru&en/v1/Books#download"



In [20]:
ext_fiction = []
opus_url = "https://object.pouta.csc.fi/OPUS-Books/v1/moses/en-ru.txt.zip"

response = requests.get(opus_url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with io.BytesIO() as buffer:
    for chunk in response.iter_content(chunk_size=4096): buffer.write(chunk)
    print("Unzipping...")
    with zipfile.ZipFile(buffer) as z:
        with z.open('Books.en-ru.ru') as f:
            for line in tqdm(f, desc="Opus Stream"):
                try:
                    t = line.decode('utf-8').strip()
                    if len(t) > 20:
                        chunks = chunk_external_text(t)
                        ext_fiction.extend(chunks)
                except: continue
print(f"Loaded {len(ext_fiction)} Opus chunks.")


Unzipping...


Opus Stream: 17496it [00:00, 53836.44it/s]

Loaded 15653 Opus chunks.


# Wikipedia

In [21]:
def check_heading_punctuation(heading_text):
    if not heading_text:
        return False
    pairs = extract_labels_standardized(heading_text)
    if not pairs:
        return False

    last_word_label = pairs[-1]['label']
    return last_word_label != LABEL_MAP["O"]

In [22]:
import re
import ast

def parse_wiki_segments(raw_item_text):
    text_content = ""
    if not isinstance(raw_item_text, str):
        try:
            text_content = raw_item_text.decode('utf-8')
        except (UnicodeDecodeError, AttributeError):
            try:
                text_content = ast.literal_eval(raw_item_text).decode('utf-8')
            except (ValueError, SyntaxError, AttributeError):
                return []
    else:
        text_content = raw_item_text

    text_content = text_content.replace('_NEWLINE_', ' ')
    text_content = re.sub(r'\s+', ' ', text_content).strip()

    tag_definitions = {
        "_START_ARTICLE_": "article_heading",
        "_START_SECTION_": "section_heading",
        "_START_PARAGRAPH_": "paragraph"
    }

    tag_pattern = '|'.join(re.escape(tag) for tag in tag_definitions.keys())

    segments = []
    last_end = 0

    first_match = re.search(tag_pattern, text_content)
    if first_match:
        pre_tag_text = text_content[0:first_match.start()]
        cleaned_pre_tag_text = robust_clean_text(pre_tag_text)
        if cleaned_pre_tag_text:
            segments.append({"text": cleaned_pre_tag_text, "type": "paragraph"})

    for match in re.finditer(tag_pattern, text_content):
        tag = match.group(0)
        start_index = match.end()

        next_match = re.search(tag_pattern, text_content[start_index:])
        if next_match:
            end_index = start_index + next_match.start()
        else:
            end_index = len(text_content)

        segment_text = text_content[start_index:end_index]
        cleaned_segment_text = robust_clean_text(segment_text)

        if cleaned_segment_text:
            segment_type = tag_definitions.get(tag, "unknown")

            if segment_type in ["article_heading", "section_heading"]:
                if not check_heading_punctuation(cleaned_segment_text):
                    continue

            segments.append({"text": cleaned_segment_text, "type": segment_type})

        last_end = end_index

    if last_end < len(text_content):
        remaining_text = text_content[last_end:]
        cleaned_remaining_text = robust_clean_text(remaining_text)
        if cleaned_remaining_text:
            segments.append({"text": cleaned_remaining_text, "type": "paragraph"})

    return segments

In [23]:
# ext_science = []
# ds_wiki = load_dataset("wiki40b", "ru", split="train", streaming=True)
# count = 0
# target_count = WIKI_LIMIT

# for item in tqdm(ds_wiki, total=target_count, desc="Wiki Stream"):
#     if count >= target_count: break

#     raw_text = item['text']
#     clean_text = fix_wiki_encoding(raw_text)
#     if len(clean_text) > 100 and "_START_" not in clean_text:
#         ext_science.append(clean_text)
#         count += 1

# print(f"Loaded {len(ext_science)} clean Wiki samples.")
# if len(ext_science) > 0:
#     print("Sample check:", ext_science[0][:100])


In [24]:
ext_science = []
ds_wiki = load_dataset("wiki40b", "ru", split="train", streaming=True)
count = 0
target_count = TARGET_PER_SOURCE

for item in tqdm(ds_wiki, total=target_count, desc="Wiki Stream"):
    if count >= target_count: break

    raw_text = item['text']
    # Use fix_wiki_encoding first to handle the raw text.
    cleaned_full_text = fix_wiki_encoding(raw_text)

    # Then parse into segments
    segments = parse_wiki_segments(cleaned_full_text)

    for segment in segments:
        text = segment['text']
        # Filter for length and append
        if len(text) > 100:
            ext_science.append(text)
            count += 1
            if count >= target_count: # Check limit after each append
                break


Resolving data files:   0%|          | 0/29 [00:00<?, ?it/s]

Wiki Stream:  47%|████▋     | 28309/60000 [00:39<00:43, 722.36it/s] 


# Merging datasets (by genres)

In [25]:
# # get Syntagrus genre tag
# def get_syn_texts(genres):
#     return df_syn[df_syn['genre'].isin(genres)]['text'].tolist()


In [26]:
# journ_texts = get_syn_texts(['journalism', 'news', 'publicistic']) + ext_journalism
# print(f"Domain 'Journalism': {len(journ_texts)} samples")

# fiction_texts = get_syn_texts(['fiction', 'literature']) + ext_fiction
# print(f"Domain 'Fiction':    {len(fiction_texts)} samples")

# science_texts = get_syn_texts(['nonfiction', 'wiki', 'science']) + ext_science
# print(f"Domain 'Science':    {len(science_texts)} samples")


# Generating jsons by genres

In [27]:
# domains = {
#     "journalism": journ_texts,
#     "fiction": fiction_texts,
#     "science": science_texts
# }

# all_train_texts = []
# all_test_texts = []

# for domain_name, texts in domains.items():
#     train_txt, test_txt = train_test_split(texts, test_size=0.1, random_state=42)

#     create_json_dataset(train_txt, f"train_{domain_name}.json")
#     create_json_dataset(test_txt, f"test_{domain_name}.json")

#     all_train_texts.extend(train_txt)
#     all_test_texts.extend(test_txt)


In [28]:
# import random
# random.shuffle(all_train_texts)

# print("\nProcessing Combined 'ALL' Dataset...")
# create_json_dataset(all_train_texts, "train_all.json")
# create_json_dataset(all_test_texts, "test_all.json")

# print("\nDONE!datasets ready for BERT:")
# print("1. train/test_all.json        (Generalist Model)")
# print("2. train/test_journalism.json (Specialist Experiment)")
# print("3. train/test_fiction.json    (Specialist Experiment)")
# print("4. train/test_science.json    (Specialist Experiment)")


# Assembling Train dataset

In [29]:
syn_texts_all = df_syn['text'].tolist()

full_corpus = syn_texts_all + ext_journalism + ext_fiction + ext_science

print(f"Source Stats:")
print(f"- SynTagRus (Train): {len(syn_texts_all)}")
print(f"- Gazeta (Journalism): {len(ext_journalism)}")
print(f"- Opus (Fiction):      {len(ext_fiction)}")
print(f"- Wiki (Science):      {len(ext_science)}")
print(f"TOTAL SAMPLES:         {len(full_corpus)}")

random.seed(42)
random.shuffle(full_corpus)

train_texts, val_texts = train_test_split(full_corpus, test_size=0.05, random_state=42)

print(f"\nTraining set size:   {len(train_texts)}")
print(f"Validation set size: {len(val_texts)}")

create_json_dataset(train_texts, "train_all.json")
create_json_dataset(val_texts, "val_internal.json")





Source Stats:
- SynTagRus (Train): 4418
- Gazeta (Journalism): 43513
- Opus (Fiction):      15653
- Wiki (Science):      60000
TOTAL SAMPLES:         123584

Training set size:   117404
Validation set size: 6180
Processing 117404 texts for train_all.json...


100%|██████████| 117404/117404 [01:02<00:00, 1890.39it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/data/train_all.json
Processing 6180 texts for val_internal.json...


100%|██████████| 6180/6180 [00:02<00:00, 2696.60it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/data/val_internal.json


# Testing data

In [30]:
syntagrus_test_url = ["https://raw.githubusercontent.com/UniversalDependencies/UD_Russian-SynTagRus/master/ru_syntagrus-ud-test.conllu"]
df_syn_test = process_syntagrus_files(syntagrus_test_url, "SynTagRus TEST")

--- PROCESSING SynTagRus TEST ---


8800it [00:06, 1427.06it/s]


In [31]:
all_test_txt = df_syn_test['text'].tolist()

create_json_dataset(all_test_txt, "bench_test_all.json")

Processing 569 texts for bench_test_all.json...


100%|██████████| 569/569 [00:00<00:00, 582.55it/s]


Saved /content/drive/MyDrive/omg_diploma_2025/data/bench_test_all.json


## *domains testing*

In [32]:
# def get_texts_by_genre(df, genres):
#     return df[df['genre'].isin(genres)]['text'].tolist()

# print("\n--- ASSEMBLING FINAL DATASETS ---")

# domains = {
#     "journalism": {
#         "syn_keywords": ['journalism', 'news', 'publicistic'],
#         "external": ext_journalism
#     },
#     "fiction": {
#         "syn_keywords": ['fiction', 'literature'],
#         "external": ext_fiction
#     },
#     "science": {
#         "syn_keywords": ['nonfiction', 'wiki', 'science'],
#         "external": ext_science
#     }
# }

In [33]:
# domain_keywords = {
#     "journalism": ['journalism', 'news', 'publicistic'],
#     "fiction": ['fiction', 'literature'],
#     "science": ['nonfiction', 'wiki', 'science']
# }

# for domain, keywords in domain_keywords.items():
#     subset = df_syn_test[df_syn_test['genre'].isin(keywords)]['text'].tolist()
#     if subset:
#         print(f"Creating specific benchmark for '{domain}' ({len(subset)} samples)...")
#         create_json_dataset(subset, f"bench_test_{domain}.json")
#     else:
#         print(f"Warning: No test samples found for domain '{domain}' (check SynTagRus tags).")


## merging training data by genres

In [34]:
# all_train_texts = []

# for domain_name, sources in domains.items():
#     syn_train_txt = get_texts_by_genre(df_syn, sources['syn_keywords'])
#     domain_train = syn_train_txt + sources['external']

#     create_json_dataset(domain_train, f"train_{domain_name}.json")
#     all_train_texts.extend(domain_train)

#     bench_test_txt = get_texts_by_genre(df_syn_test, sources['syn_keywords'])

#     if len(bench_test_txt) == 0:
#         print(f"No test data found for {domain_name} in SynTagRus Test!")
#     else:
#         create_json_dataset(bench_test_txt, f"bench_test_{domain_name}.json")

In [35]:
# random.shuffle(all_train_texts)

# # Create Validation Set (Just 5% of training data to check convergence)
# train_txt, val_txt = train_test_split(all_train_texts, test_size=0.05, random_state=42)

# print("\nProcessing Generalist Datasets...")
# create_json_dataset(train_txt, "train_all.json")
# create_json_dataset(val_txt, "val_internal.json")

# print("\nDONE! Dataset Manifest:")
# print("1. train_all.json         -> Main Training Data (Gazeta+Opus+Wiki+SynTagRusTrain)")
# print("2. val_internal.json      -> For checking loss during training")
# print("3. train_[genre].json     -> For Domain Adaptation Training")
# print("4. bench_test_[genre].json -> OFFICIAL BENCHMARK (SynTagRus Test Only)")

In [36]:
# all_bench_test_txt = df_syn_test.tolist()
# create_json_dataset(all_bench_test_txt, "bench_test_all.json")


## Visuzlization

In [37]:
data_files = {
    "train": os.path.join(output_dir, "train_all.json"),
    "validation": os.path.join(output_dir, "val_internal.json")
}
raw_datasets = load_dataset("json", data_files=data_files, streaming=True)


In [38]:
import pandas as pd

def visualize_dataset(dataset_split, num_samples=5, id2label=ID_TO_LABEL):
    samples = []
    # Iterate over the streaming dataset to collect samples
    for i, example in enumerate(dataset_split):
        if i >= num_samples:
            break
        samples.append(example)

    # If no samples were collected (e.g., empty dataset or num_samples=0)
    if not samples:
        return pd.DataFrame()

    df = pd.DataFrame(samples)
    if id2label:
        def get_label_names(tag_ids):
            return [id2label[int(t)] for t in tag_ids]

        df['ner_tags_names'] = df['ner_tags'].apply(get_label_names)
        cols = ['tokens', 'ner_tags_names', 'ner_tags'] + [c for c in df.columns if c not in ['tokens', 'ner_tags_names', 'ner_tags']]
        df = df[cols]
    pd.set_option('display.max_colwidth', None)

    return df

df_view = visualize_dataset(raw_datasets['train'], num_samples=3, id2label=ID_TO_LABEL)

display(df_view)


,tokens,ner_tags_names,ner_tags
0,"[Ахмет, Шакиров, родился, в, 1920, году, в, деревне, Бишкураево, Белебеевского, уезда, Уфимской, губернии, (ныне, Туймазинский, район, Башкортостана), в, семье, муллы, Окончил, школу-семилетку, в, которой, учился, с, 1930, года, С, 1937, по, 1939, год, учился, на, Башкирском, педагогическом, рабфаке, имени, Б, Нуриманова, в, Уфе, В, 1939, году, поступил, в, пединститут, но, в, октябре, того, же, года, был, мобилизован, в, армию, Служил, помощником, политрука, После, начала, Великой, Отечественной, войны, ушёл, на, фронт, Погиб, в, 1941, году, Начал, писать, стихи, ещё, в, школе, В, 1935, году, начал, публиковать, свои, стихи, и, репортажи, в, газете, Йәш, төҙөүсе, (, Молодой, строитель, ), Во, время, учёбы, ...]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ., O, ,, O, O, O, O, O, ., O, O, O, O, O, O, O, O, O, O, O, ., O, O, ., O, O, O, O, O, ,, O, O, O, O, O, O, O, O, O, ., O, O, ., O, O, O, O, O, O, O, ., O, O, O, ., O, O, O, O, O, ., O, O, O, O, O, O, O, O, O, O, "", O, "" , "" , O, "" , ., O, O, O, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 10, 10, 0, 10, 1, 0, 0, 0, ...]"
1,"[Также, планируется, допросить, членов, совета, при, президенте, по, содействию, развитию, институтов, гражданского, общества, и, правам, человека, и, членов, Общественной, наблюдательной, комиссии, Москвы, по, контролю, за, обеспечением, прав, человека, в, местах, принудительного, содержания, В, СКП, надеются, что, они, могли, получить, важную, для, следствия, информацию, в, ходе, своих, собственных, расследований, Сегодня, в, день, годовщины, гибели, Сергея, Магнитского, от, СКП, во, всем, мире, ждут, не, дополнительной, проверки, халатности, со, стороны, неизвестных, лиц, а, возбуждения, уголовного, дела, и, ареста, сотрудников, МВД, за, пытки, и, убийство, Магнитского, заявили, представители, фонда, Только, тогда, можно, говорить, о, реальности, намерений, СКП, расследовать, истинные, причины, гибели, Магнитского, Коллеги, погибшего, ...]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ., O, O, ,, O, O, O, O, O, O, O, O, O, O, O, O, . "", O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ,, O, O, O, O, O, O, O, O, O, O, O, O, , -, O, O, . -, O, O, O, O, O, O, O, O, O, O, O, O, ""., O, O, ...]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 27, 0, 0, 23, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 0, 0, ...]"
2,"[Дегуманизация, действия, направленные, на, лишение, субъекта, прав, принадлежности, к, человеческому, роду, В, области, социальной, психологии, дегуманизация, предполагает, действия, в, результате, которых, отдельные, социальные, группы, народы, или, расы, воспринимаются, как, неполноценные, и, недостойные, принадлежать, к, роду, человеческому, Дегуманизация, зачастую, связана, с, конфликтами, Также, наблюдается, в, обычной, жизни, как, проявление, морального, отстранения, Дегуманизация, не, всегда, связана, с, расизмом, или, ксенофобией, и, может, принимать, различные, формы, Несмотря, на, кажущуюся, распространённость, существует, множество, стратегий, по, предотвращению, дегуманизации]","[-, ,, O, O, O, O, O, O, O, O, ., O, O, O, O, O, O, ,, O, O, O, O, O, ,, O, O, ,, O, O, O, O, O, O, O, O, ., O, O, O, O, ., O, O, O, O, ,, O, O, O, ., O, O, O, O, O, O, O, O, O, O, O, O, ., O, O, O, ,, O, O, O, O, O, .]","[7, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 

In [39]:
def audit_example(dataset_split, index, id2label):
    example = dataset_split[index]
    tokens = example['tokens']
    tags = example['ner_tags']

    data = []
    for t, tag_id in zip(tokens, tags):
        data.append({
            "Token": t,
            "Tag ID": tag_id,
            "Label": id2label[int(tag_id)]
        })

    return pd.DataFrame(data)

In [40]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

def audit_example_from_dict(example_dict, id2label):
    """
    Audits a single example given as a dictionary.
    """
    tokens = example_dict['tokens']
    tags = example_dict['ner_tags']

    data = []
    for t, tag_id in zip(tokens, tags):
        data.append({
            "Token": t,
            "Tag ID": tag_id,
            "Label": id2label[int(tag_id)]
        })
    return pd.DataFrame(data)

def audit_multiple_examples(dataset_split, indices, id2label):
    collected_examples = {}
    max_index = max(indices) if indices else -1

    for i, example in enumerate(dataset_split):
        if i > max_index and len(collected_examples) == len(indices):
            break
        if i in indices:
            collected_examples[i] = example

    for idx in indices:
        if idx in collected_examples:
            print(f"\n example index = {idx}")
            df = audit_example_from_dict(collected_examples[idx], id2label)
            display(df)
        else:
            print(f"\n Example at index {idx} not found. Dataset might be smaller than requested index or not enough items were streamed.")

audit_multiple_examples(raw_datasets['train'], [0, 1, 2, 100], ID_TO_LABEL)



 example index = 0


,Token,Tag ID,Label
0,Ахмет,0,O
1,Шакиров,0,O
2,родился,0,O
3,в,0,O
4,1920,0,O
5,году,0,O
6,в,0,O
7,деревне,0,O
8,Бишкураево,0,O
9,Белебеевского,0,O



 example index = 1


,Token,Tag ID,Label
0,Также,0,O
1,планируется,0,O
2,допросить,0,O
3,членов,0,O
4,совета,0,O
5,при,0,O
6,президенте,0,O
7,по,0,O
8,содействию,0,O
9,развитию,0,O



 example index = 2


,Token,Tag ID,Label
0,Дегуманизация,7,-
1,действия,2,","
2,направленные,0,O
3,на,0,O
4,лишение,0,O
5,субъекта,0,O
6,прав,0,O
7,принадлежности,0,O
8,к,0,O
9,человеческому,0,O



 example index = 100


,Token,Tag ID,Label
0,В,0,O
1,первом,0,O
2,сете,0,O
3,довольно,0,O
4,легкая,0,O
5,победа,0,O
6,со,0,O
7,счетом,0,O
8,6,5,:
9,1,0,O
